In [ ]:
import os
os.chdir(r'C:\Users\feyza.cotuk\OneDrive - AYGAZ A.S\Tüplügaz Analiz')
import pandas as pd
import numpy as np
import geri_kalanlar.global_functionscopy as gl
import plotly.express as px
import matplotlib.pyplot as plt
import unicodedata
from unidecode import unidecode
import plotly.express as px
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.interpolate import interp1d
import plotly.graph_objects as go
import json
from collections import Counter
import re


In [2]:
DATA_PATH = r"C:\Users\feyza.cotuk\OneDrive - AYGAZ A.S\Tüplügaz Analiz\price_analytics_cylindirc"

#### *Aygaz TownID = RakipTownId eşleştirilmesi*

In [3]:
sql_query = f"""
SELECT 
    [FULL_ADDRESS],
    [PROVINCE_ID],
    [TOWN_ID]
FROM [ESSNET].[dbo].[CUSTOMER_ADDRESS_NEW]
"""
adress = gl.read_from_sql_multi(sql_query, 'ESSNET')

In [4]:
sql_query = f"""
SELECT  [Il]
      ,[Ilce]
      ,[IlKodu]
      ,[IlceKodu]
  FROM [TahminlemeDB].[dbo].[Tuplugaz_Rakip_Il_Ilce_Listesi]
    """
il_ilce = gl.read_from_sql_multi(sql_query, 'AEP')

In [5]:
adress["il"] = adress["PROVINCE_ID"].map(il_ilce[["IlKodu","Il"]].drop_duplicates().set_index("IlKodu")["Il"])
adress["ilce"] = adress["FULL_ADDRESS"].str.split("/").str[0].str.split().str[-1].str.strip()

In [6]:
il_ilce_mode = (
    adress
    .groupby(["PROVINCE_ID", "TOWN_ID"])
    .agg({
        "il": lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan,
        "ilce": lambda x: (
            m := x[~x.astype(str)
                    .str.contains("(Apt", case=False, na=False, regex=False)
                  ].mode(),
            m.iloc[0] if not m.empty else np.nan
        )[1]
    })
    .reset_index()
)
ilce_map = {
    1601: "Sarıkamış",
    1598: "Sarayönü",
    1753: "Akören",
    1933: "Güneysınır",
    1801: "Hasköy",
    1879: "Ayvacık",
    1117: "Akdağmadeni",
    1784: "Cumayeri"
}

mask = il_ilce_mode["ilce"].isna()

il_ilce_mode.loc[mask, "ilce"] = (
    il_ilce_mode.loc[mask, "TOWN_ID"].map(ilce_map)
)


In [7]:
def clean_text(s):
    if pd.isna(s):
        return s
    # Türkçe karakterleri koruyarak normalize et
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    return s

# df_clean tarafı
il_ilce_mode["ilce_clean"] = il_ilce_mode["ilce"].apply(clean_text)
il_ilce_mode["il_clean"]   = il_ilce_mode["il"].apply(clean_text)

# il_ilce tarafı
il_ilce["Ilce_clean"] = il_ilce["Ilce"].apply(clean_text)
il_ilce["Il_clean"]   = il_ilce["Il"].apply(clean_text)

# Merge işlemi temiz kolonlar üzerinden
df_merged = il_ilce_mode.merge(
    il_ilce,
    how="left",
    left_on=["PROVINCE_ID", "ilce_clean"],
    right_on=["IlKodu", "Ilce_clean"]
)


In [8]:
df_merged=df_merged[['TOWN_ID', 'PROVINCE_ID', 'IlKodu', 'IlceKodu',"Il", "Ilce"]]
df_merged["IlKodu"]=df_merged["IlKodu"].astype('Int64')
df_merged["IlceKodu"]=df_merged["IlceKodu"].astype('Int64')
df_merged.dropna(inplace=True)
df_merged.to_pickle(DATA_PATH + r"\df_ilce_eslesme.pkl")



### *Şimdilik Tüm Fonksiyonlar Burada*

In [8]:
def save_watermark(dt_str,WM_PATH):
    os.makedirs(os.path.dirname(WM_PATH), exist_ok=True)
    with open(WM_PATH, "w", encoding="utf-8") as f:
        json.dump({"last_fkdat": dt_str}, f)

def load_watermark(WM_PATH):
    if not os.path.exists(WM_PATH):
        return None
    with open(WM_PATH, "r", encoding="utf-8") as f:
        return json.load(f).get("last_fkdat")
def mode_nonzero(x):
    """0'dan farklı en çok tekrar eden değeri döner"""
    x = x[x != 0]
    m = x.mode()
    return m.iloc[0] if not m.empty else pd.NA

### *Fatura Datalarının Temizliği Ve Anlamlandırılması*
> Tüm 12 KG Ev Tüpü Faturaları çekiliyor. 

> Burda illa tüm 12 KG ları çekmek zorunda değilim fatura bilgilerini epdk,ötv,etc. gibi bilgilerin doldurulmasında kullanacağım. 

In [10]:
if not os.path.exists(DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012.parquet"):
    sql_query = f"""
    SELECT 
        [vbeln] as [talep_no]
        ,[fkdat] as [tarihfatura]
        ,[kunrg] as [kunnr]
        ,[matnr] as[mamulno]
        ,[fkimgton] as miktar_ton
        ,fkimg as adet
        ,[rafsatfyt] 
        ,[epdkkatpay] as epdk
        ,[otv] as otv 
        ,[perlistfyt] 
        ,[hesmrj] 
        ,[hesnav] 
        ,[baydgtpay] 
        ,[aygpay] 
        ,[WADAT_IST] as fiilimalcıkıs
        FROM [sqldb-aygaz-prod-001].[ods].[zaygbw_tup_ortfy]
        where matnr = 000000000006040012 
        """
    fatura = gl.read_from_sql_multi(sql_query, 'Managed_Instance_BigDataUsr')

    fatura["tarihfatura"] = pd.to_datetime(fatura["tarihfatura"], errors="coerce")
    fatura.to_parquet(DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012.parquet", index=False)

    last = fatura["tarihfatura"].max()
    save_watermark(last.strftime("%Y-%m-%d %H:%M:%S"),DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012_watermark.json")

else:
    last_fkdat = load_watermark(DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012_watermark.json")
    last_dt = pd.to_datetime(last_fkdat)
    

    sql_inc = """
    SELECT 
        [vbeln] as [talep_no],
        [fkdat] as [tarihfatura],
        [kunrg] as [kunnr],
        [matnr] as [mamulno],
        [fkimgton] as [miktar_ton],
        [fkimg] as [adet],
        [rafsatfyt],
        [epdkkatpay] as [epdk],
        [otv] as [otv],
        [perlistfyt],
        [hesmrj],
        [hesnav],
        [baydgtpay],
        [aygpay],
        [WADAT_IST] as [fiilimalcıkıs]
    FROM [sqldb-aygaz-prod-001].[ods].[zaygbw_tup_ortfy]
    WHERE matnr = 000000000006040012
    AND fkdat > ?   
    """

    df_new = gl.read_from_sql_multi(sql_inc, "Managed_Instance_BigDataUsr", params=(last_dt,))
    df_new["tarihfatura"] = pd.to_datetime(df_new["tarihfatura"], errors="coerce")

    # Eski dosya varsa oku + birleştir
    if os.path.exists(DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012.parquet"):
        df_old = pd.read_parquet(DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012.parquet")
        df_old["tarihfatura"] = pd.to_datetime(df_old["tarihfatura"], errors="coerce")
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new

    df_all.to_parquet(DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012.parquet", index=False)
    fatura=df_all.copy()
    # Yeni watermark
    new_last = df_all["tarihfatura"].max()
    save_watermark(new_last.strftime("%Y-%m-%d %H:%M:%S"), DATA_PATH + r"\zaygbw_tup_ortfy_matnr6040012_watermark.json")


In [11]:
fatura.drop(fatura[fatura["hesnav"]==0].index, inplace=True)
fatura = fatura[fatura["fiilimalcıkıs"]> '2023-01-01']
fatura.sort_values(by=["fiilimalcıkıs"], ascending=True, inplace=True)
fatura["mamulno"]=fatura["mamulno"].astype(int)
fatura["kunnr"]=(fatura["kunnr"].astype(int)).astype(str)

#### *Bayi datasının fatura ile eşleştirilmesi ; "Farklı bir db de olduğu için burada birleştiriyorum."*



In [12]:
sql_query = f"""
SELECT 
      [ERP_RETAILER_CODE]
      ,[RETAILER_NAME]
      ,[PROVINCE_ID]
      ,[TOWN_ID]
      ,COMPANY_REGION_NO
  FROM [ESSNET].[dbo].[RETAILER]
    """
bayiler = gl.read_from_sql_multi(sql_query, 'ESSNET')

In [13]:
fatura=fatura.merge(bayiler, left_on="kunnr", right_on="ERP_RETAILER_CODE", how="left", suffixes=("", "_bayi"))

In [ ]:
fatura

In [16]:
fatura["fiilimalcıkıs"]=pd.to_datetime(fatura["fiilimalcıkıs"], errors="coerce")

In [18]:
fatura.to_parquet(DATA_PATH + r"\fatura_saf.parquet", index=False)

#### *Burada tüm günler özelinde **ffill** ve **bfill** yapılarak tüm günlere epdk,ötv... bilgileri atanacak. Bu bilgiler il özelinde gruplanarak devam edilecek*

**Navlun Corrected**


In [9]:
## İl bazında navlun sabit mi ?
## Farklılık var ise max ne kadar bir fark var ?
result_hesnav = (
    fatura.groupby([fatura["fiilimalcıkıs"].dt.date, "PROVINCE_ID"])["hesnav"]
    .apply(lambda x: x.unique())  # unique hesnav değerleri
    .reset_index()
)

def max_diff(arr):
    if len(arr) > 1:
        return arr.max() - arr.min()
    else:
        return 0  

result_hesnav["max_diff"] = result_hesnav["hesnav"].apply(max_diff)

# Sıralama
result_hesnav = result_hesnav.sort_values("max_diff", ascending=False)

result_hesnav["max_diff_tl_kg"] = result_hesnav["max_diff"]/1000*12

> 1.5 Tl üstüne çalışacaım bir tutar değil o yüzden medyan alarak devam ediyorum. Önemsemiyorum. Ve eğer önceki değerle arasındaki fark 0.5 tl den az ise üstteki değer ile dolduruyorum değişikliklikleri önemsemiyorum çünkü bu bir zam değil benim için


In [10]:
result_hesnav.head(1)

,fiilimalcıkıs,PROVINCE_ID,hesnav,max_diff,max_diff_tl_kg
5592,2023-06-07,23.0,"[747.79, 747.5, 875.79]",128.29,1.53948


In [11]:
fatura['gun'] = fatura['fiilimalcıkıs'].dt.normalize()

# 2) İl + Gün bazında günlük medyan 'hesnav'
navlun_medyan = (
    fatura.groupby(['PROVINCE_ID', 'gun'], as_index=False)['hesnav']
          .median()
          .rename(columns={'hesnav': 'hesnav_corrected',"gun":"fiilimalcıkıs"})
          .sort_values(['PROVINCE_ID','fiilimalcıkıs'])
)

In [12]:
# Tarihe göre sıralama (province özelinde)
navlun_medyan = navlun_medyan.sort_values(["PROVINCE_ID","fiilimalcıkıs"])

# Province bazlı diff
navlun_medyan["hesnav_diff"] = navlun_medyan.groupby("PROVINCE_ID")["hesnav_corrected"].diff()

# 41 Tl/ton 0.5 tl ye geliyor o sebeple bunu kabul ettim
def fix_small_changes(group):
    corrected = group["hesnav_corrected"].copy()
    for i in range(1, len(group)):
        if abs(corrected.iloc[i] - corrected.iloc[i-1]) < 41:
            corrected.iloc[i] = corrected.iloc[i-1]
    group["hesnav_corrected_final"] = corrected
    return group

navlun_medyan = navlun_medyan.groupby("PROVINCE_ID", group_keys=False).apply(fix_small_changes)

C:\Users\feyza.cotuk\AppData\Local\Temp\ipykernel_10212\1181409454.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  navlun_medyan = navlun_medyan.groupby("PROVINCE_ID", group_keys=False).apply(fix_small_changes)


In [13]:
navlun_medyan

,PROVINCE_ID,fiilimalcıkıs,hesnav_corrected,hesnav_diff,hesnav_corrected_final
0,1.0,2023-01-02,577.03,NaN,577.03
1,1.0,2023-01-03,577.03,0.0,577.03
2,1.0,2023-01-04,577.03,0.0,577.03
3,1.0,2023-01-05,577.03,0.0,577.03
4,1.0,2023-01-06,577.03,0.0,577.03
...,...,...,...,...,...
41001,82.0,2026-02-10,4206.96,0.0,4206.96
41002,82.0,2026-02-12,4206.96,0.0,4206.96
41003,82.0,2026-02-13,4206.96,0.0,4206.96
41004,82.0,2026-02-16,4206.96,0.0,4206.96


> Bu aşamada şimdi eksik günleri her il için ekleyerek il-gün bazında eksiksiz bir tablo oluşturmak gerekiyor. 

In [14]:
all_days = pd.date_range(navlun_medyan['fiilimalcıkıs'].min(),
                         navlun_medyan['fiilimalcıkıs'].max(),
                         freq="D")
all_provinces = navlun_medyan['PROVINCE_ID'].unique()

# cross join → bütün kombinasyonlar
full_index = pd.MultiIndex.from_product([all_days, all_provinces],
                                        names=['fiilimalcıkıs','PROVINCE_ID'])
full_navlun_medyan = pd.DataFrame(index=full_index).reset_index()

# merge ile değerleri oturt
navlun_medyan_full = full_navlun_medyan.merge(navlun_medyan, on=['fiilimalcıkıs','PROVINCE_ID'], how='left')

navlun_medyan_full["hesnav_corrected_final"]=navlun_medyan_full["hesnav_corrected_final"].round(2)
navlun_medyan_full.sort_values(['PROVINCE_ID','fiilimalcıkıs'], inplace=True)
navlun_medyan_full

,fiilimalcıkıs,PROVINCE_ID,hesnav_corrected,hesnav_diff,hesnav_corrected_final
0,2023-01-02,1.0,577.03,NaN,577.03
82,2023-01-03,1.0,577.03,0.0,577.03
164,2023-01-04,1.0,577.03,0.0,577.03
246,2023-01-05,1.0,577.03,0.0,577.03
328,2023-01-06,1.0,577.03,0.0,577.03
...,...,...,...,...,...
93479,2026-02-14,82.0,NaN,NaN,NaN
93561,2026-02-15,82.0,NaN,NaN,NaN
93643,2026-02-16,82.0,4206.96,0.0,4206.96
93725,2026-02-17,82.0,4206.96,0.0,4206.96


> Aşağıda nan olan hesnav değerlerinde eğer önceki ve sonraki satırlarında hesnav aynı ise nan olan satırı bu hesnav değeri ile doldur

In [15]:
s = navlun_medyan_full['hesnav_corrected_final']

left  = s.groupby(navlun_medyan_full['PROVINCE_ID']).ffill()
right = s.groupby(navlun_medyan_full['PROVINCE_ID']).bfill()

mask_fill = (
    s.isna() &
    left.notna() &
    right.notna() &
    (left == right)
)

navlun_medyan_full.loc[mask_fill, 'hesnav_corrected_final'] = left[mask_fill]


> Yapılan düzeltmeye rağmen boş olan satırlar için; Eğer o gün navlun değişimi 5 den fazla il de olmuş ise ben o günü değişim günü alarak eksikleri doldurabilirim.

In [16]:
##5 den fazla ilçede o gün navlun değişmiş ise o günü al
df = navlun_medyan_full.copy() 
df['fiilimalcıkıs'] = pd.to_datetime(df['fiilimalcıkıs'], errors='coerce')
df["diff"]=df["hesnav_corrected_final"].diff()

# 1) diff>0 filtre
chg = df.loc[df['diff'] > 0, ['fiilimalcıkıs', 'PROVINCE_ID']].dropna(subset=['PROVINCE_ID'])

# (gün, il) bazında kaç defa gelmiş?
per_date_prov_counts = (
    chg.groupby(['fiilimalcıkıs','PROVINCE_ID'])
       .size()
       .rename('n')
       .reset_index()
)

# 2) o günde kaç ilde değişim olmuş?
per_date_counts = (
    per_date_prov_counts.groupby('fiilimalcıkıs')['PROVINCE_ID']
                        .nunique()
                        .rename('n_provinces')
                        .reset_index()
)

chg_dates = set(
    pd.to_datetime(
        per_date_counts.loc[per_date_counts["n_provinces"] > 5, "fiilimalcıkıs"]
    ).dt.normalize()
)

# 2) navlun_medyan_full'e bayrak sütunu ekle
navlun_medyan_full["fiyat değişim tarihleri"] = (
    navlun_medyan_full["fiilimalcıkıs"].dt.normalize().isin(chg_dates)
).astype("int8") 

In [17]:
col_date   = "fiilimalcıkıs"
col_prov   = "PROVINCE_ID"
col_value  = "hesnav_corrected_final"
col_flag   = "fiyat değişim tarihleri"

nav = navlun_medyan_full.copy()
nav[col_date]  = pd.to_datetime(nav[col_date], errors="coerce")
nav[col_value] = pd.to_numeric(nav[col_value], errors="coerce")
nav = nav.sort_values([col_prov, col_date]).reset_index(drop=True)

g = nav.groupby(col_prov, group_keys=False)

# Komşu satırların değer ve bayrakları (aynı il içinde)
prev_val = g[col_value].shift(1)
next_val = g[col_value].shift(-1)
next_fl  = g[col_flag].shift(-1)

is_nan = nav[col_value].isna()

# 1) Bu satır değişim günü ise (flag==1) -> bir SONRAKİ satırın değeriyle doldur
m1 = is_nan & (nav[col_flag] == 1)
nav.loc[m1, col_value] = next_val[m1]

# 2) Bu satır değişim günü değil ama BİR SONRAKİ satır değişim günü ise -> ÜST satırın değeriyle doldur
m2 = is_nan & (nav[col_flag] != 1) & (next_fl == 1)
nav.loc[m2, col_value] = prev_val[m2]

# (opsiyonel) sonucunu geri yaz
navlun_fixed = nav


In [18]:
def apply_change_rule(g,
                      date_col='fiilimalcıkıs',
                      val_col='hesnav_corrected_final',
                      flag_col='fiyat değişim tarihleri'):
    g = g.sort_values(date_col).reset_index(drop=True).copy()
    s = g[val_col].copy()
    flag = g[flag_col].fillna(0).astype(int)
    csum = flag.cumsum().to_numpy()

 
    if (flag == 1).any():
        first_one_pos = np.where(flag.to_numpy() == 1)[0][0]  # pozisyon
        
        before = s.iloc[:first_one_pos]
        if before.notna().any():
            base_val = before.dropna().iloc[-1]
            s.iloc[:first_one_pos] = base_val
    else:
        g[val_col] = s.ffill()
        return g

   
    max_seg = int(csum.max())
    for k in range(1, max_seg + 1):
        seg_mask = (csum == k)
        seg_vals = s[seg_mask].bfill().ffill()
        s.loc[seg_mask] = seg_vals

    g[val_col] = s
    return g

navlun_fixed = (navlun_fixed.groupby('PROVINCE_ID', group_keys=False)
        .apply(apply_change_rule))


C:\Users\feyza.cotuk\AppData\Local\Temp\ipykernel_10212\3745184433.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(apply_change_rule))


> Sadece belli başlı ilçelerde boşluk olduğu için görmezden geliyorum bu kısmı 

In [19]:
navlun_fixed[navlun_fixed["hesnav_corrected_final"].isna()]["PROVINCE_ID"].unique()

array([27., 58., 62., 66., 69., 71., 79.])

**Rafineri & EPKD & OTV** 
> *bunlar tüm türkiye için geçerli*

In [20]:
fatura["gun"] = fatura["fiilimalcıkıs"].dt.normalize()

gunluk_fark = (
    fatura.groupby("gun", as_index=False)
          .agg(
              min_raf=("rafsatfyt", "min"),
              max_raf=("rafsatfyt", "max"),
              min_epdk=("epdk", "min"),
              max_epdk=("epdk", "max"),
              min_otv=("otv", "min"),
              max_otv=("otv", "max")
          )
)
gunluk_fark["max_diff_raf"] = gunluk_fark["max_raf"] - gunluk_fark["min_raf"]
gunluk_fark["max_diff_epdk"] = gunluk_fark["max_epdk"] - gunluk_fark["min_epdk"]
gunluk_fark["max_diff_otv"] = gunluk_fark["max_otv"] - gunluk_fark["min_otv"]


>  Çok büyük diff olmadığı için önemsemiyorum bunu

In [21]:
gunluk_fark["max_diff_epdk"].max(),gunluk_fark["max_diff_epdk"].max(),gunluk_fark["max_diff_otv"].max()

(0.4200000000000017, 0.4200000000000017, 0.42000000000007276)

In [22]:
gunluk_corrected = (
    fatura
        .groupby("gun", as_index=False)
        .agg(
            epdk_medyan=("epdk", "median"),
            rafsatfyt_medyan=("rafsatfyt", "median"),
            otv_medyan=("otv", "median")
        )
        .rename(columns={"gun": "fiilimalcıkıs"})
        .sort_values("fiilimalcıkıs")
)

gunluk_corrected[["epdk_medyan", "rafsatfyt_medyan", "otv_medyan"]] = (
    gunluk_corrected[["epdk_medyan", "rafsatfyt_medyan", "otv_medyan"]].round(2)
)


In [23]:
def fix_small_changes_two(df, threshold=41,
                          cols=("epdk_medyan", "rafsatfyt_medyan", "otv_medyan"),
                          out_suffix="_corrected"):
    df = df.sort_values("fiilimalcıkıs").copy()

    for col in cols:
        corrected = df[col].copy()
        for i in range(1, len(df)):
            if pd.notna(corrected.iloc[i]) and pd.notna(corrected.iloc[i-1]):
                if abs(corrected.iloc[i] - corrected.iloc[i-1]) < threshold:
                    corrected.iloc[i] = corrected.iloc[i-1]
        df[col.replace("_medyan", out_suffix)] = corrected

    return df

gunluk_corrected = fix_small_changes_two(gunluk_corrected, threshold=41)

In [24]:
gunluk_corrected['fiilimalcıkıs'] = pd.to_datetime(gunluk_corrected['fiilimalcıkıs'])

full_range = pd.date_range(gunluk_corrected['fiilimalcıkıs'].min(),
                           gunluk_corrected['fiilimalcıkıs'].max(),
                           freq='D')

gunluk_corrected = gunluk_corrected.set_index('fiilimalcıkıs').reindex(full_range)

gunluk_corrected = gunluk_corrected.reset_index().rename(columns={'index': 'fiilimalcıkıs'})


In [25]:
def fill_constant_gaps(df, cols):
    for col in cols:
        s = df[col]

        left  = s.ffill()
        right = s.bfill()

        mask_fill = s.isna() & left.notna() & right.notna() & (left == right)
        df.loc[mask_fill, col] = left[mask_fill]

    return df

gunluk_corrected = fill_constant_gaps(
    gunluk_corrected,
    cols=["epdk_corrected", "rafsatfyt_corrected", "otv_corrected"]
)


In [26]:
gunluk_corrected[gunluk_corrected["otv_corrected"].isna()]

,fiilimalcıkıs,epdk_medyan,rafsatfyt_medyan,otv_medyan,epdk_corrected,rafsatfyt_corrected,otv_corrected
195,2023-07-16,NaN,NaN,NaN,35.67,10302.89,NaN
1095,2026-01-01,NaN,NaN,NaN,81.35,22129.17,NaN


In [27]:
gunluk_corrected.iloc[193:197]

,fiilimalcıkıs,epdk_medyan,rafsatfyt_medyan,otv_medyan,epdk_corrected,rafsatfyt_corrected,otv_corrected
193,2023-07-14,35.67,10302.89,1778.0,35.67,10302.89,1778.0
194,2023-07-15,35.67,10302.89,1778.0,35.67,10302.89,1778.0
195,2023-07-16,NaN,NaN,NaN,35.67,10302.89,NaN
196,2023-07-17,35.67,10302.89,5778.0,35.67,10302.89,5778.0


In [187]:
corrected_all = navlun_fixed.copy()

corrected_all = corrected_all.merge(gunluk_corrected, on="fiilimalcıkıs", how="left")

In [190]:
corrected_all.loc[
    corrected_all["otv_corrected"].isna() &
    (pd.to_datetime(corrected_all["fiilimalcıkıs"]) == pd.Timestamp("2026-01-01")),
    "otv_corrected"
] = 10643.3

corrected_all.loc[
    corrected_all["otv_corrected"].isna() &
    (pd.to_datetime(corrected_all["fiilimalcıkıs"]) == pd.Timestamp("2023-07-16")),
    "otv_corrected"
] = 5778.0


### *Online Satış Datalarının Temizliği Ve Anlamlandırılması*

In [29]:
il_nosu="07"

In [119]:
sql_query = f"""
SELECT 
    o.AES_ORDER_ID,
    o.INT_ORDER_CODE,
    o.LOG_CREATE_DATE as OrderDate,
    o.INT_ORDER_CODE,
    oi.ORDER_ID,
    oi.PRODUCT_ID,
    ISNULL(PER.ERP_CODE, P.ERP_PRODUCT_CODE) AS ERP_PRODUCT_CODE,
    oi.PRODUCT_NAME,
    oi.QUANTITY,
    oi.UNIT_PRICE,
    oi.DISCOUNT,
    oi.ITEM_TOTAL,
    ord.RETAILER_ID,
    ord.CUSTOMER_ID,
    ord.ORDER_SOURCE,
    ord.DELIVERY_ADDRESS,
    ca.PROVINCE_ID,
    ca.TOWN_ID,
    r.ERP_RETAILER_CODE,
    r.RETAILER_NAME,
    r.PROVINCE_ID as R_PROVINCE_ID,
    r.TOWN_ID as R_TOWN_ID
FROM [ORDER_INTERNET_SATIS] o WITH (NOLOCK)
INNER JOIN [ORDER_ITEM] oi WITH (NOLOCK) ON oi.ORDER_ID = o.[AES_ORDER_ID]
LEFT JOIN [ORDER] ord WITH (NOLOCK) ON o.[AES_ORDER_ID] = ord.ORDER_ID
LEFT JOIN (
    SELECT *
    FROM (
        SELECT *, 
               ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY LOG_CREATE_DATE DESC) AS rn
        FROM CUSTOMER_ADDRESS_NEW WITH (NOLOCK)
    ) ca_sub
    WHERE rn = 1
) ca ON ord.CUSTOMER_ID = ca.CUSTOMER_ID
LEFT JOIN RETAILER r WITH (NOLOCK) ON ord.RETAILER_ID = r.RETAILER_ID
LEFT JOIN [ESSNET].[dbo].[PRODUCT] P ON oi.PRODUCT_ID = P.PRODUCT_ID
LEFT JOIN [ESSNET].[dbo].[PRODUCT_ERP_CODE] PER ON P.PRODUCT_ID = PER.PRODUCT_ID
WHERE 
    r.PROVINCE_ID = {il_nosu}
    AND ord.ORDER_DATE >= '2023-01-01 00:00:00'
    AND ISNULL(PER.ERP_CODE, P.ERP_PRODUCT_CODE) IN (
        '6040012', '6041012', '6040212', '6041312', '6043012',
        '6041212', '6048012', '6040112', '6049012', '6041112'
    )
    """
order_online = gl.read_from_sql_multi(sql_query, 'ESSNET')
order_onlineorg=order_online.copy()

In [137]:
def extract_il_ilce(address):
    if pd.isna(address):
        return None, None

    # '/' veya ',' ayraçlarına göre böl
    parts = re.split(r"[/,]", address)
    parts = [p.strip() for p in parts if p.strip()]

    il = None
    ilce = None

    if len(parts) >= 2:
        il = parts[-1].split()[0]       
        ilce = parts[-2].split()[-1]    
    elif len(parts) == 1:
        il = parts[-1].split()[-1]      

    return il, ilce

# Uygulama
order_online[["il", "ilce"]] = order_online["DELIVERY_ADDRESS"].apply(
    lambda x: pd.Series(extract_il_ilce(x))
)

In [144]:
# Boşlukları temizle
order_online["il"] = order_online["il"].astype(str).str.strip()
order_online["ilce"] = order_online["ilce"].astype(str).str.strip()
df_merged["Il"] = df_merged["Il"].astype(str).str.strip()
df_merged["Ilce"] = df_merged["Ilce"].astype(str).str.strip()

# Eşleştirme (il + ilçe bazlı)
order_online = order_online.merge(
    df_merged,
    how="left",
    left_on=["il", "ilce"],
    right_on=["Il", "Ilce"],
    suffixes=("", "_mapped")
)

In [146]:
order_online["TOWN_ID"] = order_online["TOWN_ID"].fillna(order_online["TOWN_ID_mapped"])
order_online["PROVINCE_ID"] = order_online["PROVINCE_ID"].fillna(order_online["PROVINCE_ID_mapped"])


In [147]:
#Ürün fiyatı 0 olan satırları çıkar
order_online=order_online[order_online["ITEM_TOTAL"]!=0]
#Tavan fiyatı her zaman indirimden büyük olmalı
order_online=order_online[order_online["UNIT_PRICE"]>order_online["DISCOUNT"]]
# Tavan fiyatı ürün fiyatından büyük veya eşitdir
order_online=order_online[order_online["UNIT_PRICE"]>=order_online["ITEM_TOTAL"]]

In [151]:
order_online.drop_duplicates(inplace=True)
order_online["OrderDay"] = pd.to_datetime(order_online["OrderDate"]).dt.date

In [153]:
df_mode = (
    order_online.groupby(["OrderDay","IlceKodu"]).agg(
        unit_price_mode=("UNIT_PRICE", mode_nonzero),
        discount_mode=("DISCOUNT", mode_nonzero),
        total_quantity=("QUANTITY", "sum")   # günlük toplam satış
    )
    .reset_index()
)

> Eksiksiz bir gün-ilçe tablosu

In [156]:
towns_7_df = (
    df_merged.loc[df_merged["PROVINCE_ID"] == pd.to_numeric(il_nosu),
                  ["TOWN_ID","PROVINCE_ID","IlKodu","IlceKodu","Il","Ilce"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_mode2 = df_mode.copy()
df_mode2["OrderDay"] = pd.to_datetime(df_mode2["OrderDay"])

start_day = df_mode2["OrderDay"].min()
end_day   = df_mode2["OrderDay"].max()

days_df = pd.DataFrame({"OrderDay": pd.date_range(start_day, end_day, freq="D")})

df_grid = days_df.merge(towns_7_df, how="cross")

df_grid.head()

,OrderDay,TOWN_ID,PROVINCE_ID,IlKodu,IlceKodu,Il,Ilce
0,2023-01-01,1121,7,7,751,Antalya,Akseki
1,2023-01-01,1126,7,7,752,Antalya,Alanya
2,2023-01-01,1303,7,7,753,Antalya,Elmalı
3,2023-01-01,1333,7,7,754,Antalya,Finike
4,2023-01-01,1337,7,7,755,Antalya,Gazipaşa


In [159]:
df_online_merge=df_grid.merge(df_mode, on=["OrderDay","IlceKodu"], how="left")

In [160]:
df_online_merge = df_online_merge.sort_values(["IlceKodu", "OrderDay"])

# 2) FFill ile doldur (geçmişten)
ff_unit = df_online_merge.groupby("IlceKodu")["unit_price_mode"].ffill()
ff_disc = df_online_merge.groupby("IlceKodu")["discount_mode"].ffill()

mask = df_online_merge["unit_price_mode"].isna() | df_online_merge["discount_mode"].isna()
df_online_merge.loc[mask, "unit_price_mode"] = ff_unit.loc[mask]
df_online_merge.loc[mask, "discount_mode"]   = ff_disc.loc[mask]
df_online_merge.loc[mask, "total_quantity"]  = 0

# 3) Hâlâ NaN varsa BFill ile doldur (gelecekten)
mask2 = df_online_merge["unit_price_mode"].isna() | df_online_merge["discount_mode"].isna()

bf_unit = df_online_merge.groupby("IlceKodu")["unit_price_mode"].bfill()
bf_disc = df_online_merge.groupby("IlceKodu")["discount_mode"].bfill()

df_online_merge.loc[mask2, "unit_price_mode"] = bf_unit.loc[mask2]
df_online_merge.loc[mask2, "discount_mode"]   = bf_disc.loc[mask2]
df_online_merge.loc[mask2, "total_quantity"]  = 0

### *Online ve Fatura Datalarının Eşleştirilmesi*

In [191]:
corrected_all

,fiilimalcıkıs,PROVINCE_ID,hesnav_corrected,hesnav_diff,hesnav_corrected_final,fiyat değişim tarihleri,epdk_medyan,rafsatfyt_medyan,otv_medyan,epdk_corrected,rafsatfyt_corrected,otv_corrected
0,2023-01-02,1.0,577.03,NaN,577.03,0,35.67,11655.93,1778.0,35.67,11655.93,1778.0
1,2023-01-03,1.0,577.03,0.0,577.03,0,35.67,11655.93,1778.0,35.67,11655.93,1778.0
2,2023-01-04,1.0,577.03,0.0,577.03,0,35.67,11655.93,1778.0,35.67,11655.93,1778.0
3,2023-01-05,1.0,577.03,0.0,577.03,0,35.67,11655.93,1778.0,35.67,11655.93,1778.0
4,2023-01-06,1.0,577.03,0.0,577.03,0,35.67,11655.93,1778.0,35.67,11655.93,1778.0
...,...,...,...,...,...,...,...,...,...,...,...,...
93803,2026-02-14,82.0,NaN,NaN,4206.96,0,102.09,23186.01,11383.0,81.35,23186.01,11383.0
93804,2026-02-15,82.0,NaN,NaN,4206.96,0,NaN,NaN,NaN,81.35,23186.01,11383.0
93805,2026-02-16,82.0,4206.96,0.0,4206.96,0,102.09,23186.01,11383.0,81.35,23186.01,11383.0
93806,2026-02-17,82.0,4206.96,0.0,4206.96,0,102.09,23186.01,11383.0,81.35,23186.01,11383.0


In [192]:
merged_all_data = df_online_merge.merge(corrected_all[["hesnav_corrected_final","epdk_corrected","rafsatfyt_corrected","otv_corrected","fiilimalcıkıs","PROVINCE_ID"]], left_on=["OrderDay","PROVINCE_ID"], right_on=["fiilimalcıkıs","PROVINCE_ID"], how="left")

In [193]:
merged_all_data = merged_all_data[
    (merged_all_data["OrderDay"] != merged_all_data["OrderDay"].min()) &
    (merged_all_data["OrderDay"] != merged_all_data["OrderDay"].max())
]


> Eksik bir günün herhangi bir ilçe için olup olmadığının kontorlü


In [201]:
# Tarih kolonunu datetime'a çevir
merged_all_data["fiilimalcıkıs"] = pd.to_datetime(merged_all_data["fiilimalcıkıs"])

# Tarih aralığını kontrol et
min_date = merged_all_data["fiilimalcıkıs"].min()
max_date = merged_all_data["fiilimalcıkıs"].max()
print(f"Tarih Aralığı: {min_date} - {max_date}")

# Tüm günleri içeren bir date range oluştur
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')

# Mevcut tarihleri al
existing_dates = merged_all_data["fiilimalcıkıs"].unique()

# Eksik tarihleri bul
missing_dates = set(all_dates) - set(existing_dates)

if len(missing_dates) > 0:
    print(f"\nEksik {len(missing_dates)} gün bulundu:")
    print(sorted(missing_dates))
else:
    print("\nTüm günler mevcut, eksik tarih yok.")

# Duplicate kontrol
duplicates = merged_all_data["fiilimalcıkıs"].duplicated().sum()
print(f"\nDuplicate tarih sayısı: {duplicates}")

# Her tarihte kaç satır var
date_counts = merged_all_data.groupby("fiilimalcıkıs").size().reset_index(name='count')
print("\nTarih bazında satır sayıları:")
print(date_counts.describe())

Tarih Aralığı: 2023-01-02 00:00:00 - 2026-02-18 00:00:00

Tüm günler mevcut, eksik tarih yok.

Duplicate tarih sayısı: 20592

Tarih bazında satır sayıları:
             fiilimalcıkıs   count
count                 1144  1144.0
mean   2024-07-26 12:00:00    19.0
min    2023-01-02 00:00:00    19.0
25%    2023-10-14 18:00:00    19.0
50%    2024-07-26 12:00:00    19.0
75%    2025-05-08 06:00:00    19.0
max    2026-02-18 00:00:00    19.0
std                    NaN     0.0


In [206]:
merged_all_data.to_parquet(DATA_PATH + r"\merged_all_data.parquet", index=False)